In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, r2_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from numpy import linalg as LA, dtype
import matplotlib.pyplot as plt
import warnings;warnings.filterwarnings('ignore')
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.model_selection import GridSearchCV

In [33]:
pizza = pd.read_csv('pizza.csv')
X = pizza.drop('Sales', axis = 1)
y = pizza['Sales']

In [34]:
lr = LinearRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, X, y, cv=kfold)
print(results.mean())

0.9804940238552785


In [38]:
wiscosin = pd.read_csv('BreastCancer.csv', index_col=0)
y = wiscosin['Class']
X = wiscosin.drop('Class', axis = 1)

In [39]:
lr = LogisticRegression()
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, X, y, cv=kfold)
print(results.mean())

0.9642446043165467


In [40]:
results = cross_val_score(lr, X, y, cv=kfold, scoring='roc_auc')
print(results.mean())

0.9944643791582568


In [49]:
solver = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
scores = []
for s in solver:
    lr = LogisticRegression(solver=s)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
    results = cross_val_score(lr, X, y, cv=kfold, scoring = 'roc_auc')
    scores.append([s, results.mean()])
df_scores = pd.DataFrame(scores, columns = ['solver', 'score'])
df_scores.sort_values(by='score', ascending=False)

,solver,score
0,lbfgs,0.994464
2,newton-cg,0.994419
3,newton-cholesky,0.994419
4,sag,0.994237
1,liblinear,0.994193
5,saga,0.993694


In [48]:
solver = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
Cs = np.linspace(0.001, 4, 20)
scores = []
for s in solver:
    for c in Cs:
        lr = LogisticRegression(solver=s, C=c)
        results = cross_val_score(lr, X, y, cv=kfold, scoring = 'roc_auc')
        scores.append([s, c, results.mean()])
df_scores = pd.DataFrame(scores, columns = ['solver','C', 'score'])
df_scores.sort_values(by='score', ascending=False)

,solver,C,score
1,lbfgs,0.211474,0.994555
41,newton-cg,0.211474,0.994555
61,newton-cholesky,0.211474,0.994555
33,liblinear,2.737158,0.994554
40,newton-cg,0.001000,0.994553
...,...,...,...
113,saga,2.737158,0.993648
119,saga,4.000000,0.993648
22,liblinear,0.421947,0.993376
21,liblinear,0.211474,0.991343


# Grid Search CV

In [46]:
from sklearn.model_selection import GridSearchCV

In [47]:
params = {'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'C': np.linspace(0.001, 4, 20)}
gcv = GridSearchCV(lr, param_grid=params, cv=kfold, scoring='roc_auc')
gcv.fit(X, y)

,estimator,LogisticRegre...solver='saga')
,param_grid,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'liblinear', ...]}"
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [50]:
print(gcv.best_params_)
print(gcv.best_score_)

{'C': np.float64(0.2114736842105263), 'solver': 'lbfgs'}
0.9945549588684017


In [53]:
df_cv = pd.DataFrame(gcv.cv_results_)
df_cv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.010066,0.001049,0.003651,0.000481,0.001,lbfgs,"{'C': 0.001, 'solver': 'lbfgs'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
1,0.002857,0.000514,0.003318,0.000496,0.001,liblinear,"{'C': 0.001, 'solver': 'liblinear'}",0.935236,0.947917,0.978487,0.953353,0.978251,0.958649,0.017142,120
2,0.008477,0.000598,0.003243,0.000294,0.001,newton-cg,"{'C': 0.001, 'solver': 'newton-cg'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
3,0.004349,0.000239,0.002788,0.000149,0.001,newton-cholesky,"{'C': 0.001, 'solver': 'newton-cholesky'}",0.992754,0.995697,0.991848,0.998879,0.993590,0.994553,0.002510,5
4,0.008437,0.000456,0.002628,0.000122,0.001,sag,"{'C': 0.001, 'solver': 'sag'}",0.992754,0.995471,0.991848,0.998879,0.993590,0.994508,0.002491,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,0.003322,0.000388,0.002777,0.000258,4.000,liblinear,"{'C': 4.0, 'solver': 'liblinear'}",0.992527,0.996377,0.990263,0.999103,0.994277,0.994509,0.003053,8
116,0.009550,0.000416,0.002776,0.000238,4.000,newton-cg,"{'C': 4.0, 'solver': 'newton-cg'}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,35
117,0.005642,0.000462,0.002917,0.000181,4.000,newton-cholesky,"{'C': 4.0, 'solver': 'newton-cholesky'}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,35
118,0.010955,0.000335,0.002728,0.000247,4.000,sag,"{'C': 4.0, 'solver': 'sag'}",0.991848,0.995697,0.990489,0.998654,0.994048,0.994147,0.002877,82


# Randomized Search CV

In [54]:
from sklearn.model_selection import RandomizedSearchCV

In [56]:
params = {'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'C': np.linspace(0.001, 4, 20)}
rgcv = RandomizedSearchCV(lr, param_distributions=params, cv=kfold, scoring='roc_auc', n_iter=10, random_state=25)
rgcv.fit(X, y)

,estimator,LogisticRegre...solver='saga')
,param_distributions,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'liblinear', ...]}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,25
,error_score,nan


In [58]:
print(rgcv.best_params_)
print(rgcv.best_score_)

{'solver': 'lbfgs', 'C': np.float64(0.2114736842105263)}
0.9945549588684017


In [57]:
df_rcv = pd.DataFrame(rgcv.cv_results_)
df_rcv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_solver,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.008227,0.001470,0.004666,0.001023,newton-cholesky,0.632421,"{'solver': 'newton-cholesky', 'C': 0.632421052...",0.992754,0.996603,0.989357,0.998879,0.994277,0.994374,0.003258,6
1,0.014267,0.000454,0.003706,0.000429,saga,1.263842,"{'solver': 'saga', 'C': 1.2638421052631577}",0.988678,0.995018,0.993659,0.997982,0.993132,0.993694,0.003021,9
2,0.015637,0.002742,0.003750,0.000272,saga,2.526684,"{'solver': 'saga', 'C': 2.5266842105263154}",0.988451,0.995018,0.993659,0.997982,0.993132,0.993648,0.003096,10
3,0.011949,0.000846,0.003068,0.000197,sag,1.053368,"{'solver': 'sag', 'C': 1.0533684210526315}",0.991848,0.995697,0.990716,0.998879,0.994048,0.994237,0.002892,7
4,0.006112,0.000274,0.003006,0.000081,newton-cholesky,1.053368,"{'solver': 'newton-cholesky', 'C': 1.053368421...",0.992980,0.996830,0.989130,0.998879,0.994277,0.994419,0.003339,3
5,0.005924,0.000639,0.003003,0.000189,newton-cholesky,0.842895,"{'solver': 'newton-cholesky', 'C': 0.842894736...",0.992980,0.996830,0.989130,0.998879,0.994277,0.994419,0.003339,3
6,0.012834,0.000802,0.003414,0.000517,sag,2.526684,"{'solver': 'sag', 'C': 2.5266842105263154}",0.991848,0.995697,0.990489,0.998654,0.994048,0.994147,0.002877,8
7,0.012804,0.001061,0.003469,0.000305,lbfgs,0.211474,"{'solver': 'lbfgs', 'C': 0.2114736842105263}",0.993207,0.996603,0.989810,0.998879,0.994277,0.994555,0.003076,1
8,0.012644,0.002581,0.003921,0.000847,newton-cg,3.789526,"{'solver': 'newton-cg', 'C': 3.7895263157894736}",0.992754,0.996830,0.989130,0.998879,0.994505,0.994420,0.003360,2
9,0.011882,0.000694,0.003384,0.000301,newton-cg,0.632421,"{'solver': 'newton-cg', 'C': 0.6324210526315789}",0.992980,0.996603,0.989357,0.998879,0.994277,0.994419,0.003237,3


In [64]:
params = {'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'C': np.linspace(0.001, 4, 20)}
rgcv = RandomizedSearchCV(lr, param_distributions=params, cv=kfold, scoring='roc_auc', n_iter=20, random_state=25, verbose=3)
rgcv.fit(X, y)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.993 total time=   0.0s
[CV 2/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.997 total time=   0.0s
[CV 3/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.989 total time=   0.0s
[CV 4/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.999 total time=   0.0s
[CV 5/5] END C=0.6324210526315789, solver=newton-cholesky;, score=0.994 total time=   0.0s
[CV 1/5] END .C=1.2638421052631577, solver=saga;, score=0.989 total time=   0.0s
[CV 2/5] END .C=1.2638421052631577, solver=saga;, score=0.995 total time=   0.0s
[CV 3/5] END .C=1.2638421052631577, solver=saga;, score=0.994 total time=   0.0s
[CV 4/5] END .C=1.2638421052631577, solver=saga;, score=0.998 total time=   0.0s
[CV 5/5] END .C=1.2638421052631577, solver=saga;, score=0.993 total time=   0.0s
[CV 1/5] END .C=2.5266842105263154, solver=saga;, score=0.988 total time=   0.

,estimator,LogisticRegre...solver='saga')
,param_distributions,"{'C': array([1.0000...00000000e+00]), 'solver': ['lbfgs', 'liblinear', ...]}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,25
,error_score,nan


In [72]:
concrete  = pd.read_csv('Concrete_Data.csv')
y = concrete['Strength']
X = concrete.drop('Strength', axis = 1)

In [73]:
rf = RandomForestRegressor(random_state=25)
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 25,
 'verbose': 0,
 'warm_start': False}

In [78]:
X

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360
...,...,...,...,...,...,...,...,...
1025,276.4,116.0,90.3,179.6,8.9,870.1,768.3,28
1026,322.2,0.0,115.6,196.0,10.4,817.9,813.4,28
1027,148.5,139.4,108.6,192.7,6.1,892.4,780.0,28
1028,159.1,186.7,0.0,175.6,11.3,989.6,788.9,28


In [79]:
y

0       79.99
1       61.89
2       40.27
3       41.05
4       44.30
        ...  
1025    44.28
1026    31.18
1027    23.70
1028    32.77
1029    32.40
Name: Strength, Length: 1030, dtype: float64

In [80]:
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
scores = []
params = {'max_features': [3,4,5,6,7],
          'max_depth': [3,5,None],
          'min_samples_split': [2,5,10,20],
          'min_samples_leaf': [1,5,10,20]}
gcv = GridSearchCV(rf, param_grid=params, cv=kfold, scoring='r2', verbose=3)
gcv.fit(X, y)

Fitting 5 folds for each of 240 candidates, totalling 1200 fits
[CV 1/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=2;, score=0.636 total time=   0.1s
[CV 2/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=2;, score=0.624 total time=   0.1s
[CV 3/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=2;, score=0.671 total time=   0.1s
[CV 4/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=2;, score=0.682 total time=   0.1s
[CV 5/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=2;, score=0.694 total time=   0.1s
[CV 1/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=5;, score=0.636 total time=   0.1s
[CV 2/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=5;, score=0.624 total time=   0.1s
[CV 3/5] END max_depth=3, max_features=3, min_samples_leaf=1, min_samples_split=5;, score=0.671 total time=   0.1s
[CV 4/5] END max

,estimator,RandomForestR...ndom_state=25)
,param_grid,"{'max_depth': [3, 5, ...], 'max_features': [3, 4, ...], 'min_samples_leaf': [1, 5, ...], 'min_samples_split': [2, 5, ...]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [81]:
print(gcv.best_params_)
print(gcv.best_score_)

{'max_depth': None, 'max_features': 4, 'min_samples_leaf': 1, 'min_samples_split': 2}
0.914172402081719


In [82]:
df_gcv = pd.DataFrame(gcv.cv_results_)
df_gcv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.163622,0.008705,0.008003,0.000755,3,3,1,2,"{'max_depth': 3, 'max_features': 3, 'min_sampl...",0.636096,0.624417,0.671345,0.682100,0.693822,0.661556,0.026783,226
1,0.163436,0.004957,0.008939,0.000731,3,3,1,5,"{'max_depth': 3, 'max_features': 3, 'min_sampl...",0.636267,0.624417,0.671345,0.682100,0.693822,0.661590,0.026750,225
2,0.154748,0.005012,0.008469,0.000892,3,3,1,10,"{'max_depth': 3, 'max_features': 3, 'min_sampl...",0.636477,0.624419,0.671345,0.682055,0.693460,0.661551,0.026616,227
3,0.162523,0.006726,0.008378,0.001051,3,3,1,20,"{'max_depth': 3, 'max_features': 3, 'min_sampl...",0.636008,0.623175,0.669230,0.682011,0.692927,0.660670,0.026770,231
4,0.158763,0.003966,0.007835,0.001211,3,3,5,2,"{'max_depth': 3, 'max_features': 3, 'min_sampl...",0.635914,0.623886,0.671687,0.681413,0.693727,0.661325,0.026863,228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,0.270714,0.005733,0.008869,0.000559,None,7,10,20,"{'max_depth': None, 'max_features': 7, 'min_sa...",0.838201,0.806454,0.863947,0.887850,0.857658,0.850822,0.027268,40
236,0.235325,0.007098,0.011588,0.004861,None,7,20,2,"{'max_depth': None, 'max_features': 7, 'min_sa...",0.786552,0.752435,0.803614,0.826456,0.817721,0.797355,0.026212,115
237,0.235903,0.006712,0.008879,0.000459,None,7,20,5,"{'max_depth': None, 'max_features': 7, 'min_sa...",0.786552,0.752435,0.803614,0.826456,0.817721,0.797355,0.026212,115
238,0.238799,0.005069,0.010437,0.001234,None,7,20,10,"{'max_depth': None, 'max_features': 7, 'min_sa...",0.786552,0.752435,0.803614,0.826456,0.817721,0.797355,0.026212,115


In [85]:
test = pd.read_csv(r'C:\Users\dai\PycharmProjects\ML\Linear_Reg\testConcrete.csv')

In [88]:
best_params = gcv.best_params_
best_model = RandomForestRegressor(
    max_features=best_params['max_features'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=25
)
best_model.fit(X, y)  # Train it on your data
predictions = best_model.predict(test)
predictions

array([63.4894    , 40.6652    , 36.4088    , 43.2591    , 59.89651167,
       30.3139    , 50.965     , 61.06578667, 53.7163    , 48.1584    ,
       47.962     , 50.0491    , 49.4003    , 37.1643    ])

In [87]:
bm = gcv.best_estimator_
bm.predict(test)

array([63.4894    , 40.6652    , 36.4088    , 43.2591    , 59.89651167,
       30.3139    , 50.965     , 61.06578667, 53.7163    , 48.1584    ,
       47.962     , 50.0491    , 49.4003    , 37.1643    ])

# Serialization

In [94]:
import pickle

In [16]:
pkfile = open(r'C:\Users\dai\Downloads\Cases\Concrete_Strength\rf_conc_311.pkl', 'wb')
pickle.dump(bm, pkfile)
pkfile.close()

NameError: name 'pickle' is not defined